In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import brain_probability_flux.analyzer.hierarchical_clustering as hc
import brain_probability_flux.plotting as bpp

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT

In [ ]:
# This figure is dense enough that the project default (7pt) causes
# text/content overlap and excess empty space; use 5pt here instead.
plt.rcParams["font.size"] = 5.0

In [ ]:
_, Z = hc.linkage_matrix(
    bpp.parcel_correlation(DATA_DIR), matrix_type="precomputed"
)
surfaces = bpp.brain_surfaces()
network_ids, network_names = bpp.yeo_networks()

In [ ]:
# Dendrogram thresholds of the three groups (4, 5 and 6 clusters)
thresholds = [0.8, 0.72, 0.71]
save_dir = "../figures/fig-s04"
dendrogram_height = (max_fig_height - max_fig_height / 3 * 2) / 2
view_height = (max_fig_height - max_fig_width / 3 * 2) / 2
for threshold, labels in zip(thresholds, ["abcd", "efgh", "ijkl"]):
    fig, ax, dn = hc.plot_dendrogram(
        Z,
        no_labels=True,
        threshold=threshold,
        return_dn=True,
        draw_threshold_legend=False,
        figsize=(max_fig_width / 3 / 2, dendrogram_height),
    )
    ax.set_xlabel("Brain parcel")
    bpp.save_panel(fig, save_dir, labels[0], bbox_inches="tight")

    cluster_inds = hc.dendrogram_to_cluster_inds(dn)
    num_clusters = len(np.unique(cluster_inds))
    name = f"hierarchical{num_clusters}"
    bpp.write_surface_views(
        surfaces, cluster_inds, name, ["../figures/parcellation"]
    )
    fig, axs = plt.subplots(2, 2, figsize=(max_fig_width / 3 / 2, view_height))
    bpp.show_surface_views(axs, "../figures/parcellation", name)
    bpp.save_panel(fig, save_dir, labels[1], bbox_inches="tight")

    profile = bpp.network_profile(cluster_inds, network_ids)
    fig, axs = plt.subplots(
        1, num_clusters, figsize=(max_fig_width / 3, view_height)
    )
    bpp.plot_network_profile(axs, profile, network_names)
    fig.supxlabel("Number of overlapped brain parcels")
    bpp.save_panel(fig, save_dir, labels[2], bbox_inches="tight")

    pfa, _ = bpp.aggregated_pfa(f"threshold_{threshold:.2f}", DATA_DIR)
    color = bpp.spin_colors(num_clusters)
    pfa.perform_WPCA(spin_colors=color)
    fig, axs = plt.subplots(
        3,
        3,
        figsize=(max_fig_width / 3, max_fig_height / 3 - 0.1),
        sharex=True,
        sharey=True,
    )
    bpp.plot_pca_by_task(pfa, axs, kind="flux", scale=40, color=color)
    bpp.save_panel(fig, save_dir, labels[3], bbox_inches="tight")